# Local real-time voice agent: Colab quickstart

**Silero VAD → Parakeet-TDT STT → Qwen 4B LLM with tool calling → Kokoro TTS**, with barge-in.

1. `Runtime → Change runtime type`: pick **L4** (best value) or **A100**. A **T4** also works; the notebook switches the LLM server to Ollama + 4-bit automatically.
2. Run the cells top to bottom. The last cell prints a public `https://…trycloudflare.com` link. Open it in Chrome, **wear headphones**, and talk.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 1. Get the code

In [ ]:
# After you push this repo to GitHub, replace the URL below. Or upload a zip and unzip to /content/app.
REPO = "https://github.com/YOUR_USERNAME/local-voice-agent.git"
!git clone -q $REPO /content/app || echo "clone failed: upload the code to /content/app instead"
%cd /content/app

## 2. Install the speech stack (about 5 minutes, mostly NeMo)

In [ ]:
!apt-get -qq install -y espeak-ng zstd > /dev/null
!pip install -q -r requirements.txt

## 3. Start the LLM server

* **Ampere or newer (L4, A100, H100)**: vLLM in an **isolated venv**, so its pinned torch never fights NeMo. Prefix caching is on by default, so the system prompt and tool schemas are prefilled once and reused every turn.
* **T4 (Turing)**: Ollama with a 4-bit GGUF. vLLM's fast kernels need sm80+.

In [ ]:
import os, subprocess, time, torch, httpx

major, _ = torch.cuda.get_device_capability()
USE_VLLM = major >= 8

if USE_VLLM:
    LLM_MODEL = "Qwen/Qwen3-4B-Instruct-2507"   # safe default. To try Qwen3.5, see the last section
    TOOL_PARSER = "hermes"
    !pip install -q uv
    !uv venv -q /content/vllm-env --python 3.12
    !uv pip install -q --python /content/vllm-env/bin/python vllm
    cmd = ["/content/vllm-env/bin/vllm", "serve", LLM_MODEL, "--port", "8001",
           "--max-model-len", "8192", "--gpu-memory-utilization", "0.55",
           "--enable-auto-tool-choice", "--tool-call-parser", TOOL_PARSER]
    BASE_URL = "http://localhost:8001/v1"
else:
    LLM_MODEL = "qwen3:4b-instruct"   # check ollama.com/library/qwen3 for current tags
    !curl -fsSL https://ollama.com/install.sh | sh
    cmd = ["ollama", "serve"]
    BASE_URL = "http://localhost:11434/v1"

llm_proc = subprocess.Popen(cmd, stdout=open("/content/llm.log", "w"), stderr=subprocess.STDOUT,
                            env={**os.environ, "OLLAMA_KEEP_ALIVE": "-1"})

def wait_for(url, timeout=1200):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if httpx.get(url, timeout=2).status_code == 200:
                return print(f"ready after {time.time() - t0:.0f}s")
        except Exception:
            pass
        if llm_proc.poll() is not None:
            raise RuntimeError("LLM server exited, see /content/llm.log")
        time.sleep(3)
    raise TimeoutError(url)

wait_for(BASE_URL + "/models")
if not USE_VLLM:
    !ollama pull {LLM_MODEL}

os.environ.update(LLM_BASE_URL=BASE_URL, LLM_MODEL=LLM_MODEL)

## 4. Benchmark every stage (p50/p90 latency, STT WER, tool-call accuracy)

In [ ]:
!python -m voice_agent.bench --runs 10

## 5. Launch the voice app and get a public HTTPS link

In [ ]:
app_proc = subprocess.Popen(["python", "-m", "voice_agent.server"], stdout=open("/content/app.log", "w"),
                            stderr=subprocess.STDOUT, env=os.environ.copy())
t0 = time.time()
while "Uvicorn running" not in open("/content/app.log").read():
    if app_proc.poll() is not None or time.time() - t0 > 600:
        raise RuntimeError(open("/content/app.log").read()[-3000:])
    time.sleep(2)

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
import re
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    if m := re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line):
        print("Open this in Chrome (headphones on):", m.group(0))
        break

In [ ]:
# Live server log: latency metrics per turn, errors
!tail -n 40 /content/app.log

## Experiments

| Try | How |
|---|---|
| **Qwen3.5-4B** (newer, hybrid thinking) | In cell 3 set `LLM_MODEL="Qwen/Qwen3.5-4B"`, `TOOL_PARSER="qwen3_coder"`, then `os.environ["LLM_DISABLE_THINKING"]="1"` before the bench. Verify the parser in the model card. |
| Whisper instead of Parakeet (99 languages) | `os.environ["STT_BACKEND"]="whisper"` |
| Snappier turn-taking | `os.environ["END_SILENCE_MS"]="250"` (more false cut-offs) or add Smart Turn v3 |
| Hindi voice | `os.environ.update(TTS_LANG="h", TTS_VOICE="hf_alpha", STT_BACKEND="whisper", LANGUAGE="hi")` |
| Bigger brain on A100 | `LLM_MODEL="Qwen/Qwen3-8B"` + `LLM_DISABLE_THINKING=1` |

Restart the app after changing env vars: `app_proc.terminate()`, then re-run cell 5.